In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("GROQ_API_KEY"))

In [2]:
from ingest import load_currency_data
documents = load_currency_data()

In [3]:
import json

# Point directly to the single JSON file
file_path = './documents/currencies.json'

with open(file_path, 'r', encoding='utf-8') as f:
    documents = json.load(f)

documents

{'currencies': [{'code': 'USD',
   'name': 'US Dollar',
   'symbol': '$',
   'decimal_digits': 2,
   'countries': ['United States', 'Ecuador', 'El Salvador']},
  {'code': 'EUR',
   'name': 'Euro',
   'symbol': '€',
   'decimal_digits': 2,
   'countries': ['Germany', 'France', 'Italy', 'Spain', 'Netherlands']},
  {'code': 'GBP',
   'name': 'British Pound',
   'symbol': '£',
   'decimal_digits': 2,
   'countries': ['United Kingdom']},
  {'code': 'JPY',
   'name': 'Japanese Yen',
   'symbol': '¥',
   'decimal_digits': 0,
   'countries': ['Japan']},
  {'code': 'CHF',
   'name': 'Swiss Franc',
   'symbol': 'CHF',
   'decimal_digits': 2,
   'countries': ['Switzerland', 'Liechtenstein']},
  {'code': 'AUD',
   'name': 'Australian Dollar',
   'symbol': 'A$',
   'decimal_digits': 2,
   'countries': ['Australia']},
  {'code': 'CAD',
   'name': 'Canadian Dollar',
   'symbol': 'C$',
   'decimal_digits': 2,
   'countries': ['Canada']},
  {'code': 'CNY',
   'name': 'Chinese Yuan',
   'symbol': '¥',
 

In [4]:
from ingest import load_currency_data
documents = load_currency_data(file_path="./documents/currencies.json") # adjust path as needed

In [5]:
documents[20]

{'code': 'TRY',
 'name': 'Turkish Lira',
 'symbol': '₺',
 'decimal_digits': 2,
 'countries': ['Turkey'],
 'doc_id': 'TRY',
 'countries_text': 'Turkey'}

In [6]:
doc = documents[25]

print("Document ID:", doc["doc_id"])
print("Currency code:", doc["code"])
print("Currency name:", doc["name"])
print("Currency symbol:", doc["symbol"])
print("Countries:", doc["countries_text"])

Document ID: PHP
Currency code: PHP
Currency name: Philippine Peso
Currency symbol: ₱
Countries: Philippines


In [7]:
doc

{'code': 'PHP',
 'name': 'Philippine Peso',
 'symbol': '₱',
 'decimal_digits': 2,
 'countries': ['Philippines'],
 'doc_id': 'PHP',
 'countries_text': 'Philippines'}

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
Generate 5 natural, student-style questions based on a currency record.
Questions must be clear, concise, and answerable using the record's
fields, such as currency code, name, symbol, and countries.
Return only the questions.
""".strip()

In [10]:
data_gen_instructions

"Generate 5 natural, student-style questions based on a currency record.\nQuestions must be clear, concise, and answerable using the record's\nfields, such as currency code, name, symbol, and countries.\nReturn only the questions."

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("OPENROUTER_API_KEY"))

In [12]:
import json
user_prompt = json.dumps(doc)
user_prompt

'{"code": "PHP", "name": "Philippine Peso", "symbol": "\\u20b1", "decimal_digits": 2, "countries": ["Philippines"], "doc_id": "PHP", "countries_text": "Philippines"}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
import os
from openai import OpenAI

openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [15]:
response = openai_client.responses.create(
    model="qwen/qwen3.8-max-prime",
    input=messages,
    max_output_tokens=250,
)

print(response.output_text)

APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 250 tokens, but can only afford 196. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.', 'provider_name': None}}, 'user_id': 'user_3K5h7pZq0KX3GLiiafiS1jhQU6f'}

In [22]:
# import os
# from openai import OpenAI

# response = openai_client.responses.create(
#     model="qwen/qwen3.8-max-prime",
#     input=messages,
#     max_output_tokens=1000,
# )

# print(response.output_text)

In [26]:
def llm_structured(client, instructions, user_prompt):
    response = client.responses.create(
        model="qwen/qwen3.8-max-prime",
        instructions=instructions,
        input=user_prompt,
        max_output_tokens=200,
    )

    return response.output_text, response.usage

In [ ]:
# def llm_structured(client, instructions, user_prompt):
#     response = client.responses.create(
#         model="qwen/qwen3.8-max-prime",
#         instructions=instructions,
#         input=user_prompt,
#         max_output_tokens=400,
#     )

#     return response.output_text, response.usage

In [27]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
)

print(usage)

ResponseUsage(input_tokens=157, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=123, output_tokens_details=OutputTokensDetails(reasoning_tokens=68), total_tokens=280, cost=0.002104, is_byok=False, cost_details={'upstream_inference_cost': 0.002104, 'upstream_inference_input_cost': 0.000628, 'upstream_inference_output_cost': 0.001476})


In [28]:
from evaluation_utils import calc_price

c:\Users\Marun Yory\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
calc_price(usage)

{'input_cost': 2.355e-05, 'output_cost': 4.92e-05, 'total_cost': 7.275e-05}

In [30]:
doc

{'code': 'PHP',
 'name': 'Philippine Peso',
 'symbol': '₱',
 'decimal_digits': 2,
 'countries': ['Philippines'],
 'doc_id': 'PHP',
 'countries_text': 'Philippines'}

In [31]:
records = []

for q in result.split("\n"):
    q = q.strip().lstrip("0123456789.-) ")

    if q:
        records.append({
            "question": q,
            "document": doc["doc_id"]
        })

records

[{'question': 'What is the currency code for the Philippine Peso?',
  'document': 'PHP'},
 {'question': 'What symbol is used for the Philippine Peso?',
  'document': 'PHP'},
 {'question': 'Which country uses the Philippine Peso?', 'document': 'PHP'},
 {'question': 'How many decimal digits does the Philippine Peso have?',
  'document': 'PHP'},
 {'question': 'What is the name of the currency with the code PHP?',
  'document': 'PHP'}]

In [32]:
import pandas as pd
pd.DataFrame(records)

,question,document
0,What is the currency code for the Philippine P...,PHP
1,What symbol is used for the Philippine Peso?,PHP
2,Which country uses the Philippine Peso?,PHP
3,How many decimal digits does the Philippine Pe...,PHP
4,What is the name of the currency with the code...,PHP


In [33]:
from evaluation_utils import llm_structured_retry

In [34]:
import json
import re

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="glm-5.1"
    )

    questions = out.questions if hasattr(out, "questions") else out

    if isinstance(questions, str):
        questions = [
            re.sub(r"^\s*\d+[\.\)]\s*", "", q).strip()
            for q in questions.splitlines()
            if q.strip()
        ]

    results = [
        {
            "question": q,
            "document": doc["doc_id"]
        }
        for q in questions
    ]

    return results, usage

In [35]:
doc

{'code': 'PHP',
 'name': 'Philippine Peso',
 'symbol': '₱',
 'decimal_digits': 2,
 'countries': ['Philippines'],
 'doc_id': 'PHP',
 'countries_text': 'Philippines'}

In [36]:
generate_ground_truth(doc)

([{'question': 'What is the currency code for the Philippine Peso?',
   'document': 'PHP'},
  {'question': 'Which country uses the PHP currency?', 'document': 'PHP'},
  {'question': 'What symbol is used to represent the Philippine Peso?',
   'document': 'PHP'},
  {'question': 'How many decimal digits does the PHP currency have?',
   'document': 'PHP'},
  {'question': 'What is the official name of the currency used in the Philippines?',
   'document': 'PHP'}],
 CompletionUsage(completion_tokens=591, prompt_tokens=169, total_tokens=760, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=518, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None, video_tokens=0), cost=0.001954693, is_byok=False, cost_details={'upstream_inference_cost': 0.001954693, 'upstream_inference_prompt_cost'

In [37]:
documents[:5]

[{'code': 'USD',
  'name': 'US Dollar',
  'symbol': '$',
  'decimal_digits': 2,
  'countries': ['United States', 'Ecuador', 'El Salvador'],
  'doc_id': 'USD',
  'countries_text': 'United States, Ecuador, El Salvador'},
 {'code': 'EUR',
  'name': 'Euro',
  'symbol': '€',
  'decimal_digits': 2,
  'countries': ['Germany', 'France', 'Italy', 'Spain', 'Netherlands'],
  'doc_id': 'EUR',
  'countries_text': 'Germany, France, Italy, Spain, Netherlands'},
 {'code': 'GBP',
  'name': 'British Pound',
  'symbol': '£',
  'decimal_digits': 2,
  'countries': ['United Kingdom'],
  'doc_id': 'GBP',
  'countries_text': 'United Kingdom'},
 {'code': 'JPY',
  'name': 'Japanese Yen',
  'symbol': '¥',
  'decimal_digits': 0,
  'countries': ['Japan'],
  'doc_id': 'JPY',
  'countries_text': 'Japan'},
 {'code': 'CHF',
  'name': 'Swiss Franc',
  'symbol': 'CHF',
  'decimal_digits': 2,
  'countries': ['Switzerland', 'Liechtenstein'],
  'doc_id': 'CHF',
  'countries_text': 'Switzerland, Liechtenstein'}]

In [38]:
data_gen_instructions

"Generate 5 natural, student-style questions based on a currency record.\nQuestions must be clear, concise, and answerable using the record's\nfields, such as currency code, name, symbol, and countries.\nReturn only the questions."

In [39]:
len(documents[:5])

5

In [40]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:03<?, ?it/s]


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 1000 tokens, but can only afford 536. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.', 'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 1000 tokens, but can only afford 766. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 1000 tokens, but can only afford 536. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'Thi

In [42]:
ground_truth

[]

In [41]:
len(ground_truth)
# pd.DataFrame(ground_truth)

0

In [43]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [44]:
len(documents)

30

In [37]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(
        pool,
        documents,
        generate_ground_truth
    )

 40%|████      | 12/30 [01:12<01:49,  6.07s/it]


APIStatusError: Error code: 402 - {'error': {'message': 'This request would exceed your available credits given your current in-flight requests. Retry after in-flight requests settle, or add credits.', 'code': 402, 'metadata': {'reason': 'in_flight_budget_exhausted', 'limit_source': 'openrouter_in_flight_budget', 'remedy_hint': 'Retry after your in-flight requests settle (see the Retry-After header). Adding credits at https://openrouter.ai/settings/credits raises your in-flight budget, up to a capped ceiling.', 'headers': {'Retry-After': '120'}, 'provider_name': None}}, 'user_id': 'user_3K5h7pZq0KX3GLiiafiS1jhQU6f'}

In [ ]:
openai_client.responses.create

In [36]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

 63%|██████▎   | 19/30 [02:21<01:21,  7.43s/it]


APIStatusError: Error code: 402 - {'error': {'message': 'This request would exceed your available credits given your current in-flight requests. Retry after in-flight requests settle, or add credits.', 'code': 402, 'metadata': {'reason': 'in_flight_budget_exhausted', 'limit_source': 'openrouter_in_flight_budget', 'remedy_hint': 'Retry after your in-flight requests settle (see the Retry-After header). Adding credits at https://openrouter.ai/settings/credits raises your in-flight budget, up to a capped ceiling.', 'headers': {'Retry-After': '120'}, 'provider_name': None}}, 'user_id': 'user_3K5h7pZq0KX3GLiiafiS1jhQU6f'}